# 🛡️ SecureOps Assistant — Modular RAG System Demo

Welcome to the **SecureOps Assistant** Google Colab notebook! This system was built for the **AAI Tech Talks Hackathon 2026** to provide cybersecurity operators with an intelligent, grounded, and audit-compliant Q&A assistant for industrial control systems (ICS/OT) environments.

### 🌟 Key Features
1. **Semantic & Keyword Hybrid Search**: Combines ChromaDB dense vector retrieval (`BAAI/bge-base-en-v1.5`) with BM25 keyword search.
2. **Cross-Encoder Reranking**: Uses `cross-encoder/ms-marco-MiniLM-L-6-v2` to select the top 5 most relevant context chunks.
3. **Grounded Generation & Citation**: Utilizes `gemini-2.5-flash` with strict instructions to generate answers solely based on context, appending inline citations `[Index N]` and source logs.
4. **Honest Rejection**: Computes confidence scores via a sigmoid function on the top candidate rerank score. Automatically rejects queries outside the knowledge base if confidence is too low.

---

## 🛠️ Step 1: Environment Setup

First, we install all necessary dependencies for processing PDFs, indexing vectors, and invoking the Gemini API.

In [ ]:
# Install dependencies
!pip install -q pymupdf4llm pymupdf rank_bm25 chromadb sentence-transformers google-generativeai pandas openpyxl

## 📂 Step 2: Clone Repository & Data Ingestion

Colab needs access to the repository code and knowledge base documents (NIST PDFs and CISA CSAF JSON advisories).

In [ ]:
# Clone repository and cd into workspace
!git clone https://github.com/flower-forever/RAG-in-Automotive-Industry.git
%cd RAG-in-Automotive-Industry

## 🔑 Step 3: Configure Gemini API Key

Paste your Gemini API key from [Google AI Studio](https://aistudio.google.com/apikey). If left empty, the assistant will run in **retrieval-only warning mode**.

In [ ]:
import os
import getpass

api_key = getpass.getpass("Enter your GEMINI_API_KEY: ")
if api_key.strip():
    os.environ["GEMINI_API_KEY"] = api_key.strip()
    print("GEMINI_API_KEY configured successfully!")
else:
    print("Warning: GEMINI_API_KEY not set. Running in offline/warning mode.")

## 🗂️ Step 4: Index the Knowledge Base

Build the vector and keyword indexes. We support **Quick Mode** (parses a subset of pages for fast iteration) and **Full Mode** (parses the entire NIST PDFs and all 200 CISA CSAF advisories).

In [ ]:
from src.indexing import build_index

mode = input("Rebuild type (quick/full) [default: quick]: ").strip().lower() or "quick"
limit_pages = (mode == "quick")

print(f"Starting database build in {'Quick' if limit_pages else 'Full'} Mode...")
num_docs, num_chunks = build_index(
    csaf_dir="doc/cisa_csaf",
    csf_pdf_path="doc/NIST Cybersecurity Framework(CSF) 2.0.pdf",
    nist_pdf_path="doc/NIST.SP.800-82r3.pdf",
    db_path="chroma_db",
    collection_name="secureops_assistant",
    limit_pdf_pages=limit_pages
)
print(f"Successfully indexed {num_chunks} chunks!")

## 🔍 Step 5: Run Q&A Queries

Ask cybersecurity questions to test the retriever, Cross-Encoder reranker, and generator.

In [ ]:
from src.retrieval import SecureOpsRetriever
from src.generation import SecureOpsGenerator

# Initialize components
retriever = SecureOpsRetriever(db_path="chroma_db", collection_name="secureops_assistant")
generator = SecureOpsGenerator()

def ask_assistant(query: str, vendor: str = None, severity: str = None):
    print(f"Query: {query}")
    if vendor or severity:
        print(f"Filters: vendor={vendor}, severity={severity}")
        
    # Retrieve context
    chunks = retriever.retrieve(query, k=5, vendor=vendor, severity=severity)
    
    # Generate answer
    answer, confidence, cited = generator.generate_answer(query, chunks)
    
    print(f"\nAnswer:\n{answer}\n")
    print(f"Confidence: {confidence * 100:.1f}%")
    
    if cited:
        print("\nCited Sources:")
        for idx, doc in enumerate(cited):
            meta = doc.get("metadata", {})
            print(f"  - [{idx+1}] {meta.get('source')} | Chapter: {meta.get('chapter', 'N/A')} | Section: {meta.get('section', 'N/A')} | Page: {meta.get('page_start', '?')}")
    print("=" * 80)

# Example 1: In-domain query
ask_assistant("What network segmentation recommendations does NIST SP 800-82 Rev 3 provide?")

In [ ]:
# Example 2: Query CISA CSAF Advisories (with filter)
ask_assistant("What default credential vulnerability affects MacGregor VDR devices?", vendor="Danelec")

In [ ]:
# Example 3: Out-of-domain query (Should trigger honest rejection)
ask_assistant("What is the CEO's personal phone number?")

## 📊 Step 6: Benchmark & Run Evaluation

Run the 20-question benchmark dataset comparing Naive Vector Search (Baseline) against Advanced Hybrid Search (Advanced RAG). This generates a styled Markdown evaluation report.

In [ ]:
from src.evaluate import run_evaluation
from IPython.display import Markdown, display

# Run evaluation
run_evaluation(qa_path="data/evaluation_qa.json", db_path="chroma_db", report_output="evaluation_report.md")

# Display evaluation report
with open("evaluation_report.md", "r") as f:
    report_md = f.read()
    
display(Markdown(report_md))